# Tese PPGEE/UnB — Avaliação experimental da resiliência pós-quântica em arquiteturas IoT (ambiente Google Colab)

Notebook que executa, **em software no Colab**, os experimentos E1–E7 do Capítulo 5 e a análise estatística da tese.

**Como usar:** *Ambiente de execução → Executar tudo.* A primeira execução compila OpenSSL 3.5, liboqs e oqs-provider (≈ 10–20 min no Colab); depois é rápido. Cada sessão grava seus dados em `results/session_<data>/`, e a última seção agrega várias sessões (robustez).

| Experimento | O que mede aqui | Hipótese |
|---|---|---|
| E1/E2 | Handshake TLS 1.3 das configurações A–F e fatorial 3×4 (KEM × autenticação): computação cliente/servidor, bytes e voos medidos; transmissão **modelada** | H1, H2, H5 |
| E3 | Custo da camada de registros TLS depois do handshake (sem broker MQTT) | H1 |
| E4 | Verificação de assinatura de firmware (ECDSA, ML-DSA-44, SLH-DSA, híbrida), bytes e memória | H3 |
| E5 | Teste de troca de algoritmo (agilidade), em software | H3 |
| E6 | Continuidade: taxa de sucesso e limite inferior de Wilson sob perfis de rede **modelados** | H4 |
| E7 | Exporta as medidas e roda a sensibilidade com a sua calculadora `pqmodel` | H4 |

**Medido vs. modelado (declare isto na tese):** tempos de computação, bytes por voo, memória e tempos de verificação são **medidos** (CPU x86 do Colab, não Cortex-A53). Transmissão/espera (RTT, perda, banda, janela inicial do TCP) é **modelada** a partir dos bytes medidos. **Energia não é medida** neste ambiente.

In [ ]:
import os, sys, io, re, json, time, math, random, shutil, subprocess, platform, itertools
import datetime as dt
from pathlib import Path
import numpy as np, pandas as pd

QUICK = bool(os.environ.get("PQC_QUICK"))            # True = execução curta, só para testar o notebook
IN_COLAB = "google.colab" in sys.modules
ROOT = Path(os.environ.get("PQC_ROOT", "/content/pqc_thesis" if IN_COLAB else "./pqc_thesis")).resolve()
ROOT.mkdir(parents=True, exist_ok=True); os.chdir(ROOT); sys.path.insert(0, str(ROOT))
SESSION = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%d_%H%M%S")
RES = ROOT / "results" / f"session_{SESSION}"; (RES / "figs").mkdir(parents=True, exist_ok=True)

# ---- versões fixadas (reprodutibilidade) ------------------------------------------------
OPENSSL_VERSION = "3.5.4"
LIBOQS_REF  = "9f027e7c76ce726ba02ed6b96321e6d0f2ccfd30"
OQSPROV_REF = "69a6746cd753b7428ae163fc749b98acd56d8529"

# ---- parâmetros dos experimentos (fixe ANTES da coleta; ver Seção 5.4 da tese) -------------
SEED        = 1
N_HS        = 100 if QUICK else 1000     # handshakes por combinação KEM×autenticação
WARMUP      = 5 if QUICK else 20
BLOCK       = 25 if QUICK else 50        # tamanho do bloco; blocos são embaralhados (ordem aleatória)
N_RESUME    = 50 if QUICK else 300       # handshakes com retomada de sessão
N_APP_MSGS  = 50 if QUICK else 200       # mensagens após o handshake (E3)
FW_SIZES_KIB = [256, 1024, 4096]         # tamanhos de imagem de firmware (E4)
N_FW        = 30 if QUICK else 200       # verificações por esquema/tamanho
N_DRILL     = 5 if QUICK else 30         # trocas de algoritmo (E5)
N_UPD       = 100                        # atualizações modeladas por esquema/tamanho (E4/E6)
INTERMEDIATES = 0                        # 0 = folha assinada pela raiz; 1 = raiz -> intermediária -> folha
PHI_S       = 0.95                       # limiar de continuidade
TIMEOUT_HS_S, TIMEOUT_UPD_S = 10.0, 60.0 # limites de tempo para considerar a operação bem-sucedida

# perfis de rede (PROVISÓRIOS, como na tese): rtt_ms, perda por segmento, banda em Mbit/s
LINKS = {
    "N0": dict(rtt_ms=1,   loss=0.0,  bw_mbps=1000),
    "N1": dict(rtt_ms=40,  loss=0.01, bw_mbps=20),
    "N2": dict(rtt_ms=200, loss=0.05, bw_mbps=1),
}
# fator de escala da computação medida. 1.0 = tempos do Colab. Para aproximar um Raspberry Pi,
# meça a mesma config no Pi, calcule (mediana_Pi / mediana_Colab) e coloque aqui (ver README final).
CPU_SCALE = {"colab_x86": 1.0}

KEM = {"x25519": "X25519", "hyb": "X25519MLKEM768", "mlkem": "MLKEM768"}
AUTH = {
    "ecdsa":   dict(gen="-algorithm EC -pkeyopt ec_paramgen_curve:P-256", oqs=False, label="ECDSA P-256"),
    "mldsa44": dict(gen="-algorithm ML-DSA-44", oqs=False, label="ML-DSA-44"),
    "mldsa65": dict(gen="-algorithm ML-DSA-65", oqs=False, label="ML-DSA-65"),
    "hybrid":  dict(gen="-algorithm p256_mldsa44", oqs=True, label="ECDSA P-256 + ML-DSA-44 (oqs-provider)"),
}
# configurações do Capítulo 5: (KEM, autenticação TLS, assinatura de firmware)
CONFIGS = {
    "A": ("x25519", "ecdsa",   "ecdsa"),
    "B": ("hyb",    "ecdsa",   "ecdsa"),
    "C": ("hyb",    "hybrid",  "ecdsa"),
    "D": ("hyb",    "hybrid",  "hybrid"),
    "E": ("mlkem",  "mldsa44", "mldsa44"),
    "F": ("mlkem",  "mldsa65", "lms"),
}
print("ROOT:", ROOT, "| sessão:", SESSION, "| QUICK:", QUICK)

## 0. Ambiente da sessão
Registra o hardware do Colab (a CPU muda entre sessões e afeta os tempos) e testa se `tc netem` funciona aqui (se funcionar, a rede pode ser emulada de verdade em vez de modelada).

In [ ]:
def sh(cmd, env=None, cwd=None, log=None, check=True):
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", cwd=cwd, env=env, capture_output=True, text=True)
    if log: Path(log).write_text(r.stdout + "\n--- stderr ---\n" + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"comando falhou: {cmd}\n{r.stderr[-1500:]}")
    return r

lscpu = sh("lscpu", check=False).stdout
cpu_model = next((l.split(":",1)[1].strip() for l in lscpu.splitlines() if l.startswith("Model name")), "?")
netem = sh("tc qdisc add dev lo root netem delay 1ms 2>&1 && tc qdisc del dev lo root", check=False)
ENV = dict(session=SESSION, utc=dt.datetime.now(dt.timezone.utc).isoformat(), cpu_model=cpu_model,
           nproc=os.cpu_count(), kernel=platform.release(), python=platform.python_version(),
           colab=IN_COLAB, netem_available=(netem.returncode == 0),
           openssl_version=OPENSSL_VERSION, liboqs_ref=LIBOQS_REF, oqsprovider_ref=OQSPROV_REF,
           params=dict(N_HS=N_HS, WARMUP=WARMUP, BLOCK=BLOCK, N_RESUME=N_RESUME, N_APP_MSGS=N_APP_MSGS,
                       FW_SIZES_KIB=FW_SIZES_KIB, N_FW=N_FW, N_DRILL=N_DRILL, N_UPD=N_UPD, SEED=SEED,
                       INTERMEDIATES=INTERMEDIATES, LINKS=LINKS, CPU_SCALE=CPU_SCALE))
print("CPU:", cpu_model, "| núcleos:", os.cpu_count(), "| tc netem disponível:", ENV["netem_available"])

## 1. Compilação da pilha criptográfica
OpenSSL 3.5 (ML-KEM, ML-DSA e SLH-DSA nativos) + liboqs + oqs-provider (assinatura híbrida `p256_mldsa44`). Versões fixadas por *hash* de commit. Se já houver binários em `ROOT`, nada é recompilado.

In [ ]:
SRC = ROOT / "src"; SRC.mkdir(exist_ok=True)
OSSL = Path(os.environ.get("PQC_OSSL", ROOT / "ossl"))
OQS  = Path(os.environ.get("PQC_OQS",  ROOT / "oqs"))
PROV = Path(os.environ.get("PQC_PROV", SRC / "oqs-provider/build/lib/oqsprovider.so"))
BIN = ROOT / "bin"; BIN.mkdir(exist_ok=True)
OPENSSL = str(OSSL / "bin" / "openssl")

def build_stack():
    perl_ok = sh("perl -MText::Template -e1", check=False).returncode == 0      # exigido pelo Configure do OpenSSL 3.x
    if not (shutil.which("ninja") and shutil.which("cmake") and shutil.which("gcc") and perl_ok):
        sh("apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y ninja-build cmake build-essential perl libtext-template-perl", log=ROOT/"apt.log")
    if not (OSSL / "bin" / "openssl").exists():
        t0 = time.time(); print("compilando OpenSSL", OPENSSL_VERSION, "...")
        sh(f"cd {SRC} && curl -sSL -o openssl.tar.gz https://github.com/openssl/openssl/releases/download/openssl-{OPENSSL_VERSION}/openssl-{OPENSSL_VERSION}.tar.gz && tar xzf openssl.tar.gz", log=ROOT/"openssl_dl.log")
        sh(f"cd {SRC}/openssl-{OPENSSL_VERSION} && ./Configure --prefix={OSSL} --libdir=lib no-docs no-tests && make -j$(nproc) && make install_sw", log=ROOT/"openssl_build.log")
        print(f"  OpenSSL pronto em {time.time()-t0:.0f}s")
    if not (OQS / "lib" / "liboqs.so").exists():
        t0 = time.time(); print("compilando liboqs ...")
        sh(f"cd {SRC} && rm -rf liboqs && git init -q liboqs && cd liboqs && git remote add origin https://github.com/open-quantum-safe/liboqs.git && git fetch -q --depth 1 origin {LIBOQS_REF} && git checkout -q FETCH_HEAD", log=ROOT/"liboqs_clone.log")
        sh(f"cd {SRC}/liboqs && cmake -S . -B build -GNinja -DCMAKE_INSTALL_PREFIX={OQS} -DOQS_MINIMAL_BUILD='SIG_ml_dsa_44;SIG_ml_dsa_65;KEM_ml_kem_768' -DOQS_BUILD_ONLY_LIB=ON -DOQS_USE_OPENSSL=OFF -DBUILD_SHARED_LIBS=ON -DCMAKE_BUILD_TYPE=Release && cmake --build build && cmake --install build", log=ROOT/"liboqs_build.log")
        print(f"  liboqs pronto em {time.time()-t0:.0f}s")
    if not PROV.exists():
        t0 = time.time(); print("compilando oqs-provider ...")
        sh(f"cd {SRC} && rm -rf oqs-provider && git init -q oqs-provider && cd oqs-provider && git remote add origin https://github.com/open-quantum-safe/oqs-provider.git && git fetch -q --depth 1 origin {OQSPROV_REF} && git checkout -q FETCH_HEAD", log=ROOT/"oqsprov_clone.log")
        sh(f"cd {SRC}/oqs-provider && cmake -S . -B build -GNinja -DOPENSSL_ROOT_DIR={OSSL} -Dliboqs_DIR={OQS}/lib/cmake/liboqs -DCMAKE_PREFIX_PATH={OQS} -DCMAKE_BUILD_TYPE=Release && cmake --build build", log=ROOT/"oqsprov_build.log")
        print(f"  oqs-provider pronto em {time.time()-t0:.0f}s")

build_stack()

# arquivos de configuração do OpenSSL (o 'make install_sw' não instala openssl.cnf)
CNF_BASE = '''
[req]
distinguished_name = dn
prompt = no
x509_extensions = v3_ca
[dn]
CN = Test CA
[v3_ca]
basicConstraints = critical,CA:TRUE
keyUsage = critical,keyCertSign,cRLSign
subjectKeyIdentifier = hash
[v3_inter]
basicConstraints = critical,CA:TRUE,pathlen:0
keyUsage = critical,keyCertSign,cRLSign
[v3_leaf]
basicConstraints = CA:FALSE
subjectAltName = DNS:localhost
'''
(ROOT / "pq.cnf").write_text("openssl_conf = default_conf\n[default_conf]\nssl_conf = ssl_sect\n[ssl_sect]\nsystem_default = sd\n[sd]\n" + CNF_BASE)
(ROOT / "pq_oqs.cnf").write_text(f"openssl_conf = openssl_init\n[openssl_init]\nproviders = provider_sect\n[provider_sect]\ndefault = default_sect\noqsprovider = oqs_sect\n[default_sect]\nactivate = 1\n[oqs_sect]\nactivate = 1\nmodule = {PROV}\n" + CNF_BASE)

def env_for(use_oqs=False):
    e = dict(os.environ)
    e["OPENSSL_CONF"] = str(ROOT / ("pq_oqs.cnf" if use_oqs else "pq.cnf"))
    e["LD_LIBRARY_PATH"] = f"{OSSL}/lib:{OQS}/lib:" + e.get("LD_LIBRARY_PATH", "")
    return e

v = sh(f"{OPENSSL} version", env=env_for()).stdout.strip(); ENV["openssl_runtime"] = v
print(v); print(sh(f"{OPENSSL} list -providers", env=env_for(True)).stdout.replace("\n  ", " ").strip()[:300])

## 2. Código dos medidores
`hs_bench.c` faz o handshake TLS 1.3 cliente↔servidor **em memória** (sem rede), cronometrando separadamente cliente e servidor (tempo de CPU da *thread*, menos sensível a ruído do que o relógio) e contando bytes por voo. `sig_bench.c` mede assinatura/verificação de firmware. `pqlab.py` reúne a estatística e o modelo de enlace.

In [ ]:
%%writefile hs_bench.c
// hs_bench.c - TLS 1.3 handshake benchmark with in-memory BIO pairs (no network).
// Measures computation time of client and server separately (wall and thread-CPU),
// exact bytes per flight and optional record-layer cost after the handshake.
// Transmission time is NOT measured here: it is modeled afterwards (see notebook).
#define _GNU_SOURCE
#include <openssl/ssl.h>
#include <openssl/err.h>
#include <openssl/bio.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#include <sys/resource.h>

static double now_wall(void){struct timespec t;clock_gettime(CLOCK_MONOTONIC,&t);return t.tv_sec*1e6+t.tv_nsec/1e3;}
static double now_cpu(void){struct timespec t;clock_gettime(CLOCK_THREAD_CPUTIME_ID,&t);return t.tv_sec*1e6+t.tv_nsec/1e3;}
static void die(const char*m){fprintf(stderr,"ERROR: %s\n",m);ERR_print_errors_fp(stderr);exit(2);}


static long read_status_kb(const char*key){FILE*f=fopen("/proc/self/status","r");char line[256];long v=-1;size_t kl=strlen(key);
  if(!f)return -1;while(fgets(line,sizeof line,f)){if(!strncmp(line,key,kl)){v=atol(line+kl);break;}}fclose(f);return v;}

typedef struct { double cw,sw,cc,sc; size_t fb[8]; int nf; int ok; int reused; } res_t;

// one handshake over a fresh BIO pair; resume_sess may be NULL. Returns result; optionally returns client session.
static res_t do_hs(SSL_CTX*cctx,SSL_CTX*sctx,SSL_SESSION*resume_sess,SSL_SESSION**out_sess,SSL**keep_c,SSL**keep_s,BIO**keep_b1,BIO**keep_b2){
  res_t r;memset(&r,0,sizeof r);
  SSL*c=SSL_new(cctx),*s=SSL_new(sctx);BIO*b1,*b2;
  if(!BIO_new_bio_pair(&b1,1<<20,&b2,1<<20))die("bio pair");
  SSL_set_bio(c,b1,b1);SSL_set_bio(s,b2,b2);SSL_set_connect_state(c);SSL_set_accept_state(s);
  if(resume_sess)SSL_set_session(c,resume_sess);
  int cdone=0,sdone=0,guard=0,lastdir=-1;size_t pc=0,ps=0;
  while(!(cdone&&sdone)&&guard++<50){
    if(!cdone){
      double w0=now_wall(),c0=now_cpu();int x=SSL_do_handshake(c);double w1=now_wall(),c1=now_cpu();
      r.cw+=w1-w0;r.cc+=c1-c0;
      if(x==1)cdone=1;else{int e=SSL_get_error(c,x);if(e!=SSL_ERROR_WANT_READ&&e!=SSL_ERROR_WANT_WRITE)die("client handshake");}
      size_t tot=BIO_number_written(b1);size_t d=tot-pc;pc=tot;
      if(d>0){if(lastdir!=0){if(r.nf<8)r.nf++;lastdir=0;}r.fb[r.nf-1]+=d;}
    }
    if(!sdone){
      double w0=now_wall(),c0=now_cpu();int x=SSL_do_handshake(s);double w1=now_wall(),c1=now_cpu();
      r.sw+=w1-w0;r.sc+=c1-c0;
      if(x==1)sdone=1;else{int e=SSL_get_error(s,x);if(e!=SSL_ERROR_WANT_READ&&e!=SSL_ERROR_WANT_WRITE)die("server handshake");}
      size_t tot=BIO_number_written(b2);size_t d=tot-ps;ps=tot;
      if(d>0){if(lastdir!=1){if(r.nf<8)r.nf++;lastdir=1;}r.fb[r.nf-1]+=d;}
    }
  }
  r.ok=(cdone&&sdone);r.reused=SSL_session_reused(c);
  if(!r.ok)die("handshake did not finish");
  if(out_sess){ char b[16]; SSL_read(c,b,1); *out_sess=SSL_get1_session(c); }  // processes NewSessionTicket
  if(keep_c){*keep_c=c;*keep_s=s;*keep_b1=b1;*keep_b2=b2;}else{SSL_set_shutdown(c,SSL_SENT_SHUTDOWN|SSL_RECEIVED_SHUTDOWN);SSL_set_shutdown(s,SSL_SENT_SHUTDOWN|SSL_RECEIVED_SHUTDOWN);SSL_free(c);SSL_free(s);}
  return r;
}

int main(int argc,char**argv){
  const char *cert=NULL,*key=NULL,*ca=NULL,*groups="X25519",*sigalgs=NULL,*mode="full";
  int n=100,warm=10,msgs=0,msgsize=64;
  for(int i=1;i<argc;i++){
    if(!strcmp(argv[i],"--cert"))cert=argv[++i];
    else if(!strcmp(argv[i],"--key"))key=argv[++i];
    else if(!strcmp(argv[i],"--ca"))ca=argv[++i];
    else if(!strcmp(argv[i],"--groups"))groups=argv[++i];
    else if(!strcmp(argv[i],"--sigalgs"))sigalgs=argv[++i];
    else if(!strcmp(argv[i],"--mode"))mode=argv[++i];        // full | resume
    else if(!strcmp(argv[i],"--msgs"))msgs=atoi(argv[++i]);  // application messages after handshake
    else if(!strcmp(argv[i],"--msgsize"))msgsize=atoi(argv[++i]);
    else if(!strcmp(argv[i],"-n"))n=atoi(argv[++i]);
    else if(!strcmp(argv[i],"--warmup"))warm=atoi(argv[++i]);
  }
  if(!cert||!key||!ca){fprintf(stderr,"usage: --cert --key --ca [--groups g] [--sigalgs s] [--mode full|resume] [--msgs K --msgsize M] [-n N] [--warmup W]\n");return 1;}
  int resume=!strcmp(mode,"resume");
  SSL_CTX*sctx=SSL_CTX_new(TLS_server_method()),*cctx=SSL_CTX_new(TLS_client_method());
  if(!sctx||!cctx)die("ctx");
  SSL_CTX_set_min_proto_version(sctx,TLS1_3_VERSION);SSL_CTX_set_max_proto_version(sctx,TLS1_3_VERSION);
  SSL_CTX_set_min_proto_version(cctx,TLS1_3_VERSION);SSL_CTX_set_max_proto_version(cctx,TLS1_3_VERSION);
  if(resume){SSL_CTX_set_num_tickets(sctx,1);SSL_CTX_set_session_cache_mode(cctx,SSL_SESS_CACHE_CLIENT);}
  else{SSL_CTX_set_options(sctx,SSL_OP_NO_TICKET);SSL_CTX_set_num_tickets(sctx,0);}
  SSL_CTX_set_session_cache_mode(sctx,SSL_SESS_CACHE_OFF);SSL_CTX_set_session_id_context(sctx,(const unsigned char*)"pq",2);
  if(SSL_CTX_use_certificate_chain_file(sctx,cert)!=1)die("cert");
  if(SSL_CTX_use_PrivateKey_file(sctx,key,SSL_FILETYPE_PEM)!=1)die("key");
  if(SSL_CTX_load_verify_locations(cctx,ca,NULL)!=1)die("ca");
  SSL_CTX_set_verify(cctx,SSL_VERIFY_PEER,NULL);
  if(SSL_CTX_set1_groups_list(sctx,groups)!=1)die("server groups");
  if(SSL_CTX_set1_groups_list(cctx,groups)!=1)die("client groups");
  if(sigalgs){ if(SSL_CTX_set1_sigalgs_list(sctx,sigalgs)!=1)die("server sigalgs"); if(SSL_CTX_set1_sigalgs_list(cctx,sigalgs)!=1)die("client sigalgs"); }
  SSL_SESSION*sess=NULL;
  long rss_base=read_status_kb("VmRSS:");
  printf("iter,client_us,server_us,client_cpu_us,server_cpu_us,flights,f1_bytes,f2_bytes,f3_bytes,f4_bytes,c2s_bytes,s2c_bytes,app_c_us,app_s_us,app_overhead_bytes,resumed\n");
  for(int it=-warm;it<n;it++){
    SSL*c=NULL,*s=NULL;BIO*b1=NULL,*b2=NULL;
    if(resume){ // tickets are single-use: obtain a fresh one with an UNTIMED full handshake
      if(sess){SSL_SESSION_free(sess);sess=NULL;}
      do_hs(cctx,sctx,NULL,&sess,NULL,NULL,NULL,NULL); if(!sess)die("no session ticket obtained");
    }
    res_t r=do_hs(cctx,sctx,resume?sess:NULL,NULL,msgs?&c:NULL,msgs?&s:NULL,msgs?&b1:NULL,msgs?&b2:NULL);
    double ac=0,as=0,ovh=0;
    if(msgs){
      char*buf=malloc(msgsize),*rb=malloc(msgsize);memset(buf,0x5a,msgsize);
      size_t w0=BIO_number_written(b1);
      for(int k=0;k<msgs;k++){
        double a=now_cpu();if(SSL_write(c,buf,msgsize)!=msgsize)die("app write");ac+=now_cpu()-a;
        a=now_cpu();if(SSL_read(s,rb,msgsize)!=msgsize)die("app read");as+=now_cpu()-a;
      }
      ovh=((double)(BIO_number_written(b1)-w0))/msgs-msgsize;
      ac/=msgs;as/=msgs;free(buf);free(rb);SSL_set_shutdown(c,SSL_SENT_SHUTDOWN|SSL_RECEIVED_SHUTDOWN);SSL_set_shutdown(s,SSL_SENT_SHUTDOWN|SSL_RECEIVED_SHUTDOWN);SSL_free(c);SSL_free(s);
    }
    size_t c2s=0,s2c=0;
    // direction-resolved totals: flights alternate c,s,c,...
    for(int i=0;i<r.nf;i++){ if(i%2==0)c2s+=r.fb[i]; else s2c+=r.fb[i]; }
    if(it>=0)printf("%d,%.1f,%.1f,%.1f,%.1f,%d,%zu,%zu,%zu,%zu,%zu,%zu,%.2f,%.2f,%.1f,%d\n",it,r.cw,r.sw,r.cc,r.sc,r.nf,r.fb[0],r.fb[1],r.fb[2],r.fb[3],c2s,s2c,ac,as,ovh,r.reused);
  }
  fprintf(stderr,"peak_rss_kb=%ld rss_base_kb=%ld\n",read_status_kb("VmHWM:"),rss_base);
  return 0;
}

In [ ]:
%%writefile sig_bench.c
// sig_bench.c - sign/verify benchmark for firmware signature schemes (EVP).
#define _GNU_SOURCE
#include <openssl/evp.h>
#include <openssl/pem.h>
#include <openssl/err.h>
#include <openssl/rand.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#include <sys/resource.h>
static double now_cpu(void){struct timespec t;clock_gettime(CLOCK_THREAD_CPUTIME_ID,&t);return t.tv_sec*1e6+t.tv_nsec/1e3;}

static long read_status_kb(const char*key){FILE*f=fopen("/proc/self/status","r");char line[256];long v=-1;size_t kl=strlen(key);
  if(!f)return -1;while(fgets(line,sizeof line,f)){if(!strncmp(line,key,kl)){v=atol(line+kl);break;}}fclose(f);return v;}
static void die(const char*m){fprintf(stderr,"ERROR: %s\n",m);ERR_print_errors_fp(stderr);exit(2);}
int main(int argc,char**argv){
  const char*key=NULL,*md=NULL;size_t size=1<<20;int n=100,warm=5;
  for(int i=1;i<argc;i++){
    if(!strcmp(argv[i],"--key"))key=argv[++i];
    else if(!strcmp(argv[i],"--md"))md=argv[++i];   // omit for pure (ML-DSA, SLH-DSA)
    else if(!strcmp(argv[i],"--size"))size=strtoull(argv[++i],0,10);
    else if(!strcmp(argv[i],"-n"))n=atoi(argv[++i]);
    else if(!strcmp(argv[i],"--warmup"))warm=atoi(argv[++i]);
  }
  if(!key){fprintf(stderr,"usage: --key priv.pem [--md sha256] [--size bytes] [-n N]\n");return 1;}
  long rss_base=read_status_kb("VmRSS:");
  BIO*bf=BIO_new_file(key,"r");if(!bf)die("open key");
  EVP_PKEY*pk=PEM_read_bio_PrivateKey(bf,NULL,NULL,NULL);if(!pk)die("read key");
  unsigned char*msg=malloc(size);RAND_bytes(msg,(int)size);
  const EVP_MD*mdp=md?EVP_get_digestbyname(md):NULL;
  EVP_MD_CTX*ctx=EVP_MD_CTX_new();size_t siglen=0;
  if(EVP_DigestSignInit(ctx,NULL,mdp,NULL,pk)!=1)die("signinit");
  if(EVP_DigestSign(ctx,NULL,&siglen,msg,size)!=1)die("siglen");
  unsigned char*sig=malloc(siglen);
  double t0=now_cpu();
  if(EVP_DigestSign(ctx,sig,&siglen,msg,size)!=1)die("sign");
  double sign_us=now_cpu()-t0;EVP_MD_CTX_free(ctx);
  printf("# sig_bytes=%zu sign_us=%.1f msg_bytes=%zu\niter,verify_us\n",siglen,sign_us,size);
  for(int it=-warm;it<n;it++){
    EVP_MD_CTX*v=EVP_MD_CTX_new();double a=now_cpu();
    if(EVP_DigestVerifyInit(v,NULL,mdp,NULL,pk)!=1)die("verifyinit");
    int r=EVP_DigestVerify(v,sig,siglen,msg,size);double b=now_cpu();
    if(r!=1)die("verify failed");
    if(it>=0)printf("%d,%.1f\n",it,b-a);
    EVP_MD_CTX_free(v);
  }
  fprintf(stderr,"peak_rss_kb=%ld rss_base_kb=%ld\n",read_status_kb("VmHWM:"),rss_base);
  return 0;
}

In [ ]:
%%writefile pqlab.py
"""pqlab - helpers for the PQC-IoT thesis experiments (Colab notebook).

Contents: Wilson lower bound, bootstrap CI, TCP/link model for the transmission part of a
handshake, pairwise statistics with Holm correction, factorial ANOVA with partial eta-squared.
Everything that is *modeled* (not measured) lives in this file and is labeled as such.
"""
import math
import numpy as np
import pandas as pd
from scipy import stats

# ----------------------------------------------------------------------------- continuity
def wilson_lower(k, n, z=1.96):
    """Lower limit of the Wilson score interval for k successes in n trials."""
    if n == 0:
        return 0.0
    p = k / n
    d = 1 + z * z / n
    centre = p + z * z / (2 * n)
    adj = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return max(0.0, (centre - adj) / d)

# ----------------------------------------------------------------------------- bootstrap
def bootstrap_ci(x, stat=np.median, B=10000, alpha=0.05, seed=1):
    x = np.asarray(x, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(B, len(x)))
    s = stat(x[idx], axis=1)
    return float(stat(x)), float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2))

# ----------------------------------------------------------------------------- link model (MODELED)
MSS = 1460          # bytes of TCP payload per segment
INIT_CWND = 10      # segments (RFC 6928)
RTO_MIN_US = 200_000
SYN_RTO_US = 1_000_000

def _rto_us(rtt_us):
    return max(RTO_MIN_US, 2 * rtt_us)

def flight_time_us(nbytes, link, rng):
    """Time to deliver one handshake flight of `nbytes` over a TCP connection in slow start.
    link = dict(rtt_ms, loss, bw_mbps). Losses cost one RTO each window that suffers any loss."""
    rtt = link["rtt_ms"] * 1000.0
    p = link["loss"]
    bw = link["bw_mbps"] * 1e6 / 8.0  # bytes/s
    segs = max(1, math.ceil(nbytes / MSS))
    cwnd, sent, t = INIT_CWND, 0, 0.0
    while sent < segs:
        w = min(cwnd, segs - sent)
        t += w * MSS / bw * 1e6                      # serialization of the window
        if p > 0 and rng.binomial(w, p) > 0:          # at least one loss in the window
            t += _rto_us(rtt)
        sent += w
        if sent < segs:
            t += rtt                                  # wait for ACKs before the next window
            cwnd *= 2
    return t + rtt / 2.0                              # one-way propagation

def tx_time_us(flight_bytes, link, rng, tcp_setup=True):
    """Modeled transmission/waiting time of a TLS 1.3 handshake (3 flights) on `link`."""
    rtt = link["rtt_ms"] * 1000.0
    t = 0.0
    if tcp_setup:
        t += rtt
        while rng.random() < link["loss"]:            # lost SYN: 1 s timeout
            t += SYN_RTO_US
    for b in flight_bytes[:3]:
        t += flight_time_us(b, link, rng)
    return t

def add_modeled_times(df, links, cpu_scale=None, seed=1, timeout_s=10.0):
    """Adds modeled link columns to a handshake DataFrame (one row per handshake and link).
    cpu_scale: dict profile->factor applied to MEASURED computation (default 1.0 = this machine).
    Returns long DataFrame with columns tx_us, total_us, ok (completed within timeout)."""
    cpu_scale = cpu_scale or {"R0": 1.0}
    rng = np.random.default_rng(seed)
    rows = []
    for lname, link in links.items():
        for prof, k in cpu_scale.items():
            for r in df.itertuples(index=False):
                fb = [r.f1_bytes, r.f2_bytes, r.f3_bytes]
                tx = tx_time_us(fb, link, rng)
                dev = r.client_cpu_us * k
                srv = r.server_cpu_us
                total = tx + dev + srv
                rows.append((r.config, r.iter, lname, prof, dev, srv, tx, tx + srv, total,
                             total <= timeout_s * 1e6))
    return pd.DataFrame(rows, columns=["config", "iter", "net", "profile", "dev_comp_us", "srv_comp_us",
                                       "tx_us", "wait_us", "total_us", "ok"])

# ----------------------------------------------------------------------------- statistics
def holm(pvals):
    p = np.asarray(pvals, dtype=float)
    order = np.argsort(p)
    m = len(p)
    adj = np.empty(m)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj

def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    u = stats.mannwhitneyu(a, b, alternative="two-sided").statistic
    return 2 * u / (len(a) * len(b)) - 1

def compare_to_baseline(df, group_col, metric, baseline, B=5000, seed=1):
    """Median + bootstrap CI per group, delta vs baseline, Mann-Whitney with Holm, Cliff's delta."""
    base = df.loc[df[group_col] == baseline, metric].to_numpy()
    out, praw = [], []
    for g, sub in df.groupby(group_col, sort=True):
        x = sub[metric].to_numpy()
        med, lo, hi = bootstrap_ci(x, B=B, seed=seed)
        if g == baseline:
            out.append(dict(group=g, n=len(x), median=med, ci_lo=lo, ci_hi=hi, delta_vs_base=0.0, p=np.nan, cliffs_delta=0.0))
            praw.append(np.nan)
            continue
        p = stats.mannwhitneyu(x, base, alternative="two-sided").pvalue
        out.append(dict(group=g, n=len(x), median=med, ci_lo=lo, ci_hi=hi, delta_vs_base=med - float(np.median(base)),
                        p=p, cliffs_delta=cliffs_delta(x, base)))
        praw.append(p)
    res = pd.DataFrame(out)
    mask = res["p"].notna()
    res.loc[mask, "p_holm"] = holm(res.loc[mask, "p"].to_numpy())
    return res

def kruskal_with_posthoc(df, group_col, metric):
    groups = [g[metric].to_numpy() for _, g in df.groupby(group_col)]
    h, p = stats.kruskal(*groups)
    names = sorted(df[group_col].unique())
    pairs, pv = [], []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a = df.loc[df[group_col] == names[i], metric]
            b = df.loc[df[group_col] == names[j], metric]
            pairs.append((names[i], names[j]))
            pv.append(stats.mannwhitneyu(a, b, alternative="two-sided").pvalue)
    adj = holm(pv)
    post = pd.DataFrame([(a, b, p0, p1) for (a, b), p0, p1 in zip(pairs, pv, adj)], columns=["g1", "g2", "p", "p_holm"])
    return dict(H=h, p=p), post

def factorial_anova(df, metric, f1="kem", f2="auth", log=True):
    """Two-factor ANOVA on log(metric); returns table with partial eta-squared per term."""
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    d = df.copy()
    d["y"] = np.log(d[metric]) if log else d[metric]
    model = smf.ols(f"y ~ C({f1}) * C({f2})", data=d).fit()
    tab = sm.stats.anova_lm(model, typ=2)
    ss_res = tab.loc["Residual", "sum_sq"]
    tab["partial_eta2"] = tab["sum_sq"] / (tab["sum_sq"] + ss_res)
    tab.loc["Residual", "partial_eta2"] = np.nan
    return tab

def permutation_anova_effect(df, metric, f1="kem", f2="auth", B=2000, seed=1):
    """Permutation p-values for the main effects (permuting the OTHER factor's residual structure is
    not trivial; here we permute the factor labels within the levels of the other factor)."""
    rng = np.random.default_rng(seed)
    d = df.copy()
    d["y"] = np.log(d[metric])
    def eta(dd, f, other):
        grand = dd["y"].mean()
        ss_f = sum(len(g) * (g["y"].mean() - grand) ** 2 for _, g in dd.groupby(f))
        ss_t = ((dd["y"] - grand) ** 2).sum()
        return ss_f / ss_t
    res = {}
    for f, o in ((f1, f2), (f2, f1)):
        obs = eta(d, f, o)
        cnt = 0
        for _ in range(B):
            dd = d.copy()
            dd[f] = dd.groupby(o)[f].transform(lambda s: rng.permutation(s.values))
            if eta(dd, f, o) >= obs:
                cnt += 1
        res[f] = dict(eta2=obs, p_perm=(cnt + 1) / (B + 1))
    return res

In [ ]:
for src, out in (("hs_bench.c", "hs_bench"), ("sig_bench.c", "sig_bench")):
    sh(f"gcc -O2 -o {BIN/out} {src} -I{OSSL}/include -L{OSSL}/lib -lssl -lcrypto -Wl,-rpath,{OSSL}/lib")
import importlib, pqlab; importlib.reload(pqlab)
TASKSET = ["taskset", "-c", str(os.cpu_count() - 1)] if shutil.which("taskset") else []   # fixa em um núcleo
print("harness compilado; fixando CPU:", TASKSET or "não")

## 3. Certificados
Uma PKI por algoritmo de autenticação (raiz como âncora de confiança; a cadeia enviada contém a folha e, se `INTERMEDIATES=1`, a intermediária).

In [ ]:
PKI = ROOT / "pki"; PKI.mkdir(exist_ok=True)
def make_pki(name):
    d = PKI / name; d.mkdir(exist_ok=True); e = env_for(AUTH[name]["oqs"]); g = AUTH[name]["gen"]; cnf = ROOT / ("pq_oqs.cnf" if AUTH[name]["oqs"] else "pq.cnf")
    O = OPENSSL
    sh(f"{O} genpkey {g} -out root.key && {O} req -x509 -new -key root.key -subj '/CN=Root {name}' -days 60 -out root.pem", cwd=d, env=e)
    issuer_crt, issuer_key, chain = "root.pem", "root.key", ""
    if INTERMEDIATES:
        sh(f"{O} genpkey {g} -out inter.key && {O} req -new -key inter.key -subj '/CN=Inter {name}' -out inter.csr && "
           f"{O} x509 -req -in inter.csr -CA root.pem -CAkey root.key -CAcreateserial -days 60 -extfile {cnf} -extensions v3_inter -out inter.crt", cwd=d, env=e)
        issuer_crt, issuer_key, chain = "inter.crt", "inter.key", "inter.crt"
    sh(f"{O} genpkey {g} -out leaf.key && {O} req -new -key leaf.key -subj '/CN=localhost' -out leaf.csr && "
       f"{O} x509 -req -in leaf.csr -CA {issuer_crt} -CAkey {issuer_key} -CAcreateserial -days 60 -extfile {cnf} -extensions v3_leaf -out leaf.crt", cwd=d, env=e)
    sh(f"cat leaf.crt {chain} > chain.pem", cwd=d)
    return d
PKI_OK = {}
for a in AUTH:
    try: make_pki(a); PKI_OK[a] = True
    except Exception as ex: PKI_OK[a] = False; print("PKI falhou para", a, "->", str(ex)[:200])
print(PKI_OK)

## 4. Piloto
Verifica quais combinações executam. Uma combinação que **não executa** não é descartada: o modelo atribui $F_c = 0$ e o fato é um resultado (Seção 5.3 da tese).

In [ ]:
import pqlab
def run_hs(kem, auth, n, warmup=WARMUP, mode="full", msgs=0, msgsize=64, pin=True):
    d = PKI / auth
    cmd = [str(BIN/"hs_bench"), "--cert", str(d/"chain.pem"), "--key", str(d/"leaf.key"), "--ca", str(d/"root.pem"),
           "--groups", KEM[kem], "-n", str(n), "--warmup", str(warmup), "--mode", mode]
    if msgs: cmd += ["--msgs", str(msgs), "--msgsize", str(msgsize)]
    r = subprocess.run((TASKSET if pin else []) + cmd, env=env_for(AUTH[auth]["oqs"]), capture_output=True, text=True)
    if r.returncode != 0: return None, r.stderr.strip()[-300:]
    df = pd.read_csv(io.StringIO(r.stdout))
    m = re.search(r"peak_rss_kb=(\d+) rss_base_kb=(\d+)", r.stderr)
    return df, dict(peak_kb=int(m.group(1)), base_kb=int(m.group(2))) if m else dict(peak_kb=np.nan, base_kb=np.nan)

FACT = [(k, a) for k in KEM for a in AUTH]            # fatorial 3 x 4 = 12 combinações
pilot = []
for k, a in FACT:
    df, info = run_hs(k, a, 3, warmup=1) if PKI_OK.get(a) else (None, "PKI ausente")
    pilot.append(dict(kem=k, auth=a, executable=df is not None, note="" if df is not None else info,
                      s2c_bytes=None if df is None else int(df.s2c_bytes.median()),
                      c2s_bytes=None if df is None else int(df.c2s_bytes.median())))
pilot = pd.DataFrame(pilot); pilot.to_csv(RES / "pilot.csv", index=False)
OK = {(r.kem, r.auth) for r in pilot.itertuples() if r.executable}
display(pilot)

## 5. E1 e E2 — Handshake TLS 1.3 (A–F e fatorial 3×4)
Coleta em **blocos embaralhados** (ordem aleatória, como exige a Seção 5.4), com *warm-up* descartado a cada bloco. A configuração D tem o mesmo handshake que C (diferem só na assinatura de firmware).

In [ ]:
BLOCKS = N_HS // BLOCK
tasks = [(k, a, b) for (k, a) in FACT if (k, a) in OK for b in range(BLOCKS)]
random.Random(SEED).shuffle(tasks)
frames, rss = [], {}
t0 = time.time()
for i, (k, a, b) in enumerate(tasks):
    df, info = run_hs(k, a, BLOCK)
    if df is None: raise RuntimeError(f"{k}+{a}: {info}")
    df["kem"], df["auth"], df["block"] = k, a, b
    df["iter"] = b * BLOCK + df["iter"]
    frames.append(df)
    if (k, a) not in rss or info["peak_kb"] > rss[(k, a)]["peak_kb"]: rss[(k, a)] = info
    if (i + 1) % max(1, len(tasks) // 5) == 0: print(f"  {i+1}/{len(tasks)} blocos ({time.time()-t0:.0f}s)")
hs = pd.concat(frames, ignore_index=True)
hs["combo"] = hs.kem + "+" + hs.auth
hs["comp_us"] = hs.client_cpu_us + hs.server_cpu_us
hs.to_csv(RES / "hs_raw.csv", index=False)

# configurações A-F a partir das combinações medidas (C e D compartilham o handshake)
hs_cfg = pd.concat([hs[(hs.kem == k) & (hs.auth == a)].assign(config=c) for c, (k, a, _) in CONFIGS.items() if (k, a) in OK], ignore_index=True)
print(len(hs), "handshakes;", hs.combo.nunique(), "combinações")
summ = hs_cfg.groupby("config").agg(c2s_B=("c2s_bytes","median"), s2c_B=("s2c_bytes","median"), voos=("flights","median"),
        cliente_us=("client_cpu_us","median"), servidor_us=("server_cpu_us","median"), n=("iter","count"))
summ["pico_RSS_MB"] = [rss[(CONFIGS[c][0], CONFIGS[c][1])]["peak_kb"] / 1024 for c in summ.index]
summ["RSS_extra_MB"] = [(rss[(CONFIGS[c][0], CONFIGS[c][1])]["peak_kb"] - rss[(CONFIGS[c][0], CONFIGS[c][1])]["base_kb"]) / 1024 for c in summ.index]   # crescimento durante os handshakes
display(summ.round(1))

### 5.1 Tempo total com transmissão modelada, H1 e H5
Tempo total = transmissão modelada (TCP com janela inicial de 10 segmentos, RTT, perda e banda) + computação medida. **Espera** = transmissão + tempo do servidor (definição da tese: total − computação do dispositivo).

In [ ]:
mt = pqlab.add_modeled_times(hs_cfg, LINKS, CPU_SCALE, seed=SEED, timeout_s=TIMEOUT_HS_S)
mt.to_csv(RES / "hs_modeled.csv", index=False)
prof = list(CPU_SCALE)[0]
rows = []
for net in LINKS:
    sub = mt[(mt.net == net) & (mt.profile == prof)]
    t = pqlab.compare_to_baseline(sub, "config", "total_us", "A").assign(net=net, metric="total_us")
    rows.append(t)
cmp_total = pd.concat(rows); display(cmp_total.round(3))

# H1: aumento de espera/transmissão > aumento de computação, em relação à configuração A
h1 = []
for net in LINKS:
    sub = mt[(mt.net == net) & (mt.profile == prof)].groupby("config").agg(comp=("dev_comp_us","median"), wait=("wait_us","median"))
    for c in sub.index.drop("A"):
        dc, dw = sub.loc[c, "comp"] - sub.loc["A", "comp"], sub.loc[c, "wait"] - sub.loc["A", "wait"]
        h1.append(dict(net=net, config=c, d_comp_us=dc, d_wait_us=dw, H1_suportada=bool(dw > dc)))
h1 = pd.DataFrame(h1); display(h1.round(1))
print("H1 (N0 reportado separadamente, conforme a tese):")
print(h1.groupby("net").H1_suportada.mean().round(2).to_string())

In [ ]:
import matplotlib; import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
order = list(CONFIGS)
s = hs_cfg.groupby("config")[["c2s_bytes", "s2c_bytes"]].median().reindex(order)
ax[0].bar(order, s.c2s_bytes, label="cliente → servidor"); ax[0].bar(order, s.s2c_bytes, bottom=s.c2s_bytes, label="servidor → cliente")
ax[0].set_title("Bytes do handshake (mediana)"); ax[0].set_ylabel("bytes"); ax[0].legend()
c = hs_cfg.groupby("config")[["client_cpu_us", "server_cpu_us"]].median().reindex(order) / 1000
ax[1].bar(order, c.client_cpu_us, label="cliente (dispositivo)"); ax[1].bar(order, c.server_cpu_us, bottom=c.client_cpu_us, label="servidor (gateway)")
ax[1].set_title("Computação medida (mediana)"); ax[1].set_ylabel("ms"); ax[1].legend()
w = 0.27
for i, net in enumerate(LINKS):
    sub = mt[(mt.net == net) & (mt.profile == prof)].groupby("config").total_us.median().reindex(order) / 1000
    ax[2].bar(np.arange(len(order)) + (i - 1) * w, sub.values, w, label=net)
ax[2].set_xticks(range(len(order))); ax[2].set_xticklabels(order); ax[2].set_yscale("log")
ax[2].set_title("Tempo total do handshake (modelado)"); ax[2].set_ylabel("ms (log)"); ax[2].legend()
plt.tight_layout(); plt.savefig(RES / "figs" / "e1_handshake.png", dpi=160); plt.show()

### 5.2 E2 — Fatorial KEM × autenticação e H2
ANOVA fatorial sobre o logaritmo da métrica; compara-se a variância explicada ($\eta^2$ parcial) de cada fator. H2 é suportada se a **autenticação** explicar mais variância do que o **KEM**. A energia não é medida; usa-se tempo de CPU como *proxy* e isso deve ser declarado.

In [ ]:
fact = hs[hs.combo.isin([f"{k}+{a}" for k, a in OK])].copy()
res = {}
res["computação (cliente+servidor)"] = pqlab.factorial_anova(fact, "comp_us")
# tempo total modelado em cada perfil de rede (uma linha por handshake)
fm = fact.assign(config=fact.combo)
mf = pqlab.add_modeled_times(fm, LINKS, CPU_SCALE, seed=SEED)
mf["kem"] = mf.config.str.split("+").str[0]; mf["auth"] = mf.config.str.split("+").str[1]
for net in LINKS:
    res[f"tempo total modelado {net}"] = pqlab.factorial_anova(mf[mf.net == net], "total_us")
h2 = []
for k, t in res.items():
    e_k, e_a = t.loc["C(kem)", "partial_eta2"], t.loc["C(auth)", "partial_eta2"]
    h2.append(dict(metrica=k, eta2_kem=e_k, eta2_auth=e_a, H2_suportada=bool(e_a > e_k)))
h2 = pd.DataFrame(h2)
mag = lambda e: "desprezível" if e < 0.01 else "pequeno" if e < 0.06 else "médio" if e < 0.14 else "grande"
h2["magnitude_kem"], h2["magnitude_auth"] = h2.eta2_kem.map(mag), h2.eta2_auth.map(mag)
display(h2.round(3))
print("Leia H2 junto com a magnitude: com η² < 0,01 nos dois fatores, 'suportada' não tem relevância prática.")
# bytes são determinísticos (variância ~0): em vez de ANOVA, compara-se o efeito de cada fator nos bytes
bm = fact.groupby(["kem", "auth"]).s2c_bytes.median().unstack("auth")
ef_kem  = (bm.mean(axis=1).max() - bm.mean(axis=1).min()); ef_auth = (bm.mean(axis=0).max() - bm.mean(axis=0).min())
print(f"Bytes servidor→cliente: amplitude do efeito do KEM = {ef_kem:.0f} B; da autenticação = {ef_auth:.0f} B")
display(bm.astype(int))
h2.to_csv(RES / "h2_anova.csv", index=False)

### 5.3 Retomada de sessão e custo após o handshake (E3)
A retomada (PSK) elimina a verificação de certificados: mostra quanto do custo pós-quântico é do handshake completo. Em E3, as mensagens após o handshake usam criptografia simétrica (mantida constante), então o custo por mensagem deve ser o mesmo em todas as configurações. **Não há broker MQTT neste ambiente**: mede-se a camada de registros TLS.

In [ ]:
rs = []
for k, a in FACT:
    if (k, a) not in OK: continue
    df, info = run_hs(k, a, N_RESUME, mode="resume")
    if df is None: continue
    df["kem"], df["auth"] = k, a; rs.append(df)
resume = pd.concat(rs, ignore_index=True); resume["combo"] = resume.kem + "+" + resume.auth
assert resume.resumed.all(), "algum handshake não foi retomado"
resume.to_csv(RES / "resume_raw.csv", index=False)
full = hs.groupby("combo")[["client_cpu_us", "server_cpu_us", "c2s_bytes", "s2c_bytes"]].median()
res_m = resume.groupby("combo")[["client_cpu_us", "server_cpu_us", "c2s_bytes", "s2c_bytes"]].median()
cmp_r = (res_m / full).round(2).add_suffix("_retomada/completo"); display(cmp_r)

app = []
for c, (k, a, _) in CONFIGS.items():
    if (k, a) not in OK: continue
    for ms in (64, 1024):
        df, info = run_hs(k, a, 20, warmup=3, msgs=N_APP_MSGS, msgsize=ms)
        if df is None: continue
        app.append(dict(config=c, msg_bytes=ms, cliente_us_msg=df.app_c_us.median(), servidor_us_msg=df.app_s_us.median(), overhead_B=df.app_overhead_bytes.median()))
app = pd.DataFrame(app); app.to_csv(RES / "app_data.csv", index=False); display(app.round(2))

## 6. E4 — Verificação e atualização de firmware
Esquemas: ECDSA P-256 (SHA-256), ML-DSA-44, SLH-DSA-SHA2-128s e a combinação ECDSA + ML-DSA-44 (as duas assinaturas devem ser válidas). Mede-se verificação, tamanho da assinatura/chave e pico de memória. O tempo de transferência da imagem é **modelado** pelo enlace. **LMS não está incluído** (o OpenSSL 3.5 só verifica LMS; a assinatura exige a API *stateful* da liboqs): a configuração F fica com o firmware LMS como *não medido*.

In [ ]:
FW = RES / "fw"; FW.mkdir(exist_ok=True)
e0 = env_for(False)
sh(f"{OPENSSL} genpkey -algorithm EC -pkeyopt ec_paramgen_curve:P-256 -out {FW}/ecdsa.key", env=e0)
sh(f"{OPENSSL} genpkey -algorithm ML-DSA-44 -out {FW}/mldsa44.key", env=e0)
sh(f"{OPENSSL} genpkey -algorithm SLH-DSA-SHA2-128s -out {FW}/slh.key", env=e0)
def pk_bytes(name): return len(subprocess.run(f"{OPENSSL} pkey -in {FW}/{name}.key -pubout -outform DER", shell=True, env=e0, capture_output=True).stdout)
KEYB = {n: pk_bytes(n) for n in ("ecdsa", "mldsa44", "slh")}

def run_sig(name, md, size, n):
    cmd = TASKSET + [str(BIN/"sig_bench"), "--key", str(FW/f"{name}.key"), "--size", str(size), "-n", str(n)] + (["--md", md] if md else [])
    r = subprocess.run(cmd, env=e0, capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(r.stderr[-300:])
    L = r.stdout.strip().splitlines(); meta = dict(kv.split("=") for kv in L[0][2:].split())
    m = re.search(r"peak_rss_kb=(\d+) rss_base_kb=(\d+)", r.stderr)
    return pd.read_csv(io.StringIO("\n".join(L[1:]))).verify_us.to_numpy(), float(meta["sign_us"]), int(meta["sig_bytes"]), int(m.group(1)), int(m.group(2))

fw_rows, fw_ver = [], {}
for kib in FW_SIZES_KIB:
    size = kib * 1024
    for name, md_ in (("ecdsa", "sha256"), ("mldsa44", None), ("slh", None)):
        ver, sign_us, sig_b, rss_, rssb_ = run_sig(name, md_, size, N_FW)
        signs = [sign_us] + [run_sig(name, md_, size, 1)[1] for _ in range(2)]
        fw_ver[(name, kib)] = ver
        fw_rows.append(dict(scheme=name, size_kib=kib, verify_med_us=np.median(ver), sign_med_us=np.median(signs), sig_bytes=sig_b, pubkey_bytes=KEYB[name], rss_peak_kb=rss_, rss_extra_kb=rss_ - rssb_))
    v = fw_ver[("ecdsa", kib)] + fw_ver[("mldsa44", kib)]            # híbrida: as duas verificações
    r1 = [r for r in fw_rows if r["scheme"] == "ecdsa" and r["size_kib"] == kib][0]; r2 = [r for r in fw_rows if r["scheme"] == "mldsa44" and r["size_kib"] == kib][0]
    fw_ver[("hybrid", kib)] = v
    fw_rows.append(dict(scheme="hybrid", size_kib=kib, verify_med_us=np.median(v), sign_med_us=r1["sign_med_us"] + r2["sign_med_us"],
                        sig_bytes=r1["sig_bytes"] + r2["sig_bytes"], pubkey_bytes=KEYB["ecdsa"] + KEYB["mldsa44"], rss_peak_kb=max(r1["rss_peak_kb"], r2["rss_peak_kb"]), rss_extra_kb=max(r1["rss_extra_kb"], r2["rss_extra_kb"])))
fw = pd.DataFrame(fw_rows); fw.to_csv(RES / "fw_metrics.csv", index=False); display(fw.round(1))

# tempo de atualização modelado = transferência (imagem + assinatura) + verificação medida
rng = np.random.default_rng(SEED); upd = []
for (scheme, kib), ver in fw_ver.items():
    sig_b = int(fw[(fw.scheme == scheme) & (fw.size_kib == kib)].sig_bytes.iloc[0])
    for net, link in LINKS.items():
        for u in range(N_UPD):
            t = pqlab.flight_time_us(kib * 1024 + sig_b, link, rng) + float(rng.choice(ver))
            upd.append((scheme, kib, net, t, t <= TIMEOUT_UPD_S * 1e6))
upd = pd.DataFrame(upd, columns=["scheme", "size_kib", "net", "total_us", "ok"]); upd.to_csv(RES / "fw_update_modeled.csv", index=False)
display(upd.groupby(["scheme", "size_kib", "net"]).total_us.median().unstack("net").div(1e6).round(2).rename_axis(columns="tempo (s)"))

## 7. E5 — Teste de troca de algoritmo (agilidade, em software)
Para cada transição (ex.: A→B, C→D), reconfigura-se o serviço para a nova configuração e verifica-se se o primeiro handshake é bem-sucedido e quanto leva (início do processo + carga de chaves + handshake). É uma checagem **funcional em software**: a troca real via atualização no dispositivo não é testada aqui.

In [ ]:
drill = []
for src, dst in (("A", "B"), ("B", "E"), ("C", "D"), ("D", "E")):
    kd, ad, _ = CONFIGS[dst]
    for _ in range(N_DRILL):
        t0 = time.perf_counter(); df, info = run_hs(kd, ad, 1, warmup=0, pin=False); dt_ = (time.perf_counter() - t0) * 1e3
        drill.append(dict(transicao=f"{src}→{dst}", sucesso=df is not None, tempo_ms=dt_))
drill = pd.DataFrame(drill); drill.to_csv(RES / "drill.csv", index=False)
display(drill.groupby("transicao").agg(sucesso=("sucesso", "mean"), mediana_ms=("tempo_ms", "median"), n=("sucesso", "count")).round(2))

## 8. E6 — Continuidade
Sucesso = operação concluída dentro do limite de tempo, sob os perfis de rede modelados. Indicador: limite inferior de Wilson (z = 1,96). Com $\varphi_S = 0{,}95$, são necessárias ≥ 73 tentativas todas bem-sucedidas.

In [ ]:
rows = []
for (cfg, net), g in mt[mt.profile == prof].groupby(["config", "net"]):
    k, n = int(g.ok.sum()), len(g); rows.append(dict(operacao="handshake", config_ou_esquema=cfg, net=net, sucessos=k, tentativas=n, wilson_inf=pqlab.wilson_lower(k, n)))
for (sch, net), g in upd.groupby(["scheme", "net"]):
    k, n = int(g.ok.sum()), len(g); rows.append(dict(operacao="atualização", config_ou_esquema=sch, net=net, sucessos=k, tentativas=n, wilson_inf=pqlab.wilson_lower(k, n)))
cont = pd.DataFrame(rows); cont["atinge_limiar"] = cont.wilson_inf >= PHI_S; cont.to_csv(RES / "continuity.csv", index=False)
display(cont.pivot_table(index=["operacao", "config_ou_esquema"], columns="net", values="wilson_inf").round(3))
print("Tentativas mínimas para atingir", PHI_S, "com 100% de sucesso:", next(n for n in range(1, 500) if pqlab.wilson_lower(n, n) >= PHI_S))

## 9. E7 — Sensibilidade (calculadora `pqmodel`)
Exporta as **medianas medidas** por configuração em JSON. Para rodar a análise de sensibilidade, suba o código do Apêndice B (pasta `pqmodel/`) para `ROOT` e adapte o JSON de entrada ao esquema do `profile.json` da calculadora (os nomes dos campos dependem do seu Apêndice B/Capítulo 4).

In [ ]:
meas = {}
for c, (k, a, f) in CONFIGS.items():
    if (k, a) not in OK: meas[c] = dict(executable=False); continue
    g = hs_cfg[hs_cfg.config == c]
    meas[c] = dict(executable=True, firmware_signature=f,
        handshake=dict(device_comp_us=float(g.client_cpu_us.median()), server_comp_us=float(g.server_cpu_us.median()),
                       c2s_bytes=float(g.c2s_bytes.median()), s2c_bytes=float(g.s2c_bytes.median()), flights=float(g.flights.median()),
                       peak_rss_kb=float(rss[(k, a)]["peak_kb"]), rss_extra_kb=float(rss[(k, a)]["peak_kb"] - rss[(k, a)]["base_kb"])),
        continuity={n: dict(successes=int(x.sucessos), trials=int(x.tentativas)) for n, x in cont[(cont.operacao == "handshake") & (cont.config_ou_esquema == c)].set_index("net").iterrows()},
        firmware=None if f == "lms" else {str(r.size_kib): dict(verify_us=r.verify_med_us, sig_bytes=int(r.sig_bytes), rss_peak_kb=r.rss_peak_kb, rss_extra_kb=r.rss_extra_kb) for r in fw[fw.scheme == f].itertuples()})
(RES / "measurements_for_model.json").write_text(json.dumps(meas, indent=2))
try:
    import pqmodel
    print("pqmodel encontrado:", pqmodel.__file__, "— rode: python -m pqmodel profile.json --sens 10000 --seed 1")
except ImportError:
    print("pqmodel não encontrado em", ROOT, "— faça upload do Apêndice B e use measurements_for_model.json como base do profile.json")

## 10. Robustez entre sessões
A CPU do Colab muda entre sessões. Rode o notebook em **várias sessões** (idealmente em horários diferentes); esta célula agrega todas as sessões em `results/` e mede a variação entre elas. Variação grande entre sessões deve entrar em *Ameaças à Validade*.

In [ ]:
files = sorted((ROOT / "results").glob("session_*/hs_raw.csv"))
allhs = pd.concat([pd.read_csv(f).assign(session=f.parent.name) for f in files], ignore_index=True)
print("sessões agregadas:", len(files))
if len(files) >= 2:
    med = allhs.groupby(["combo", "session"]).comp_us.median().unstack("session")
    cv = (med.std(axis=1) / med.mean(axis=1) * 100).rename("CV entre sessões (%)")
    from scipy import stats
    kw = {c: stats.kruskal(*[g.comp_us.to_numpy() for _, g in d.groupby("session")]).pvalue for c, d in allhs.groupby("combo")}
    display(pd.concat([med.round(0), cv.round(1), pd.Series(kw, name="Kruskal p (sessões)").round(4)], axis=1))
else:
    print("Só há uma sessão. Rode o notebook de novo em outra sessão do Colab para medir a variação entre sessões.")

## 11. Salvar resultados

In [ ]:
(RES / "environment.json").write_text(json.dumps(ENV, indent=2, default=str))
zipf = shutil.make_archive(str(ROOT / f"results_{SESSION}"), "zip", RES)
print("resultados:", RES); print("zip:", zipf)
if IN_COLAB:
    try:
        from google.colab import drive, files
        drive.mount("/content/drive"); dst = Path("/content/drive/MyDrive/tese_pqc_results"); dst.mkdir(exist_ok=True)
        shutil.copy(zipf, dst); print("copiado para o Google Drive:", dst)
    except Exception as e:
        print("Drive não montado:", e)

## Limitações (para a seção *Ameaças à Validade*)
- **Hardware:** tempos de CPU x86 compartilhada do Colab, não Cortex-A53. Use `CPU_SCALE` com um fator calibrado no Raspberry Pi para aproximar, e declare que é uma aproximação.
- **Rede e energia:** transmissão/espera é modelada (TCP simplificado); energia não é medida.
- **MQTT:** E3 mede a camada de registros TLS, sem broker.
- **Firmware:** LMS não incluído; atualização medida como transferência modelada + verificação medida, sem dispositivo.
- **Bibliotecas de pesquisa:** o oqs-provider/liboqs não são recomendados para produção.